# Fase 2 · Clasificador de postura hawkish/dovish (Bloque 2)

Benchmark de modelos para clasificar la postura de política monetaria de cada frase de las ruedas de prensa del BCE. Sigue la plantilla común de benchmarks: objetivo, candidatos, datos de referencia, métrica, resultados, latencia y memoria, y decisión.

## 1. Objetivo

Clasificar cada frase del panel (declaración inicial y respuestas de presidenta y vicepresidente) según la orientación de política monetaria que transmite: **hawkish**, **neutral** o **dovish**. El modelo debe devolver probabilidades por clase: la puntuación de cada frase es `p_hawkish − p_dovish`, en [−1, 1], y la de cada rueda de prensa es su media sobre las frases relevantes. Las frases sin contenido de política monetaria (saludos, procedimiento) se marcan como no relevantes y no cuentan en la puntuación.

El benchmark responde a dos preguntas: qué modelo clasifica mejor el lenguaje de las ruedas de prensa del BCE y con qué coste computacional, ya que el modelo elegido se aplicará a unas 12.000 frases del histórico.

## 2. Candidatos

| Candidato | Tipo | Tamaño | Licencia | Ejecución | Qué aporta a la comparación |
|---|---|---|---|---|---|
| `gtfintechlab/model_european_central_bank_stance_label` | RoBERTa ajustado con frases del BCE (WCB) | 0,4B | CC BY-NC-SA 4.0 | Local | Especialista del dominio |
| `gtfintechlab/model_WCB_stance_label` | RoBERTa ajustado con frases de 25 bancos centrales (WCB) | 0,4B | CC BY-NC-SA 4.0 | Local | ¿Aporta generalizar entre bancos centrales? |
| `gtfintechlab/FOMC-RoBERTa` | RoBERTa-large ajustado con frases de la Fed | 0,4B | CC BY-NC 4.0 | Local | Transferencia de la Fed al BCE |
| `BAAI/bge-large-en-v1.5` + regresión logística | Embeddings preentrenados y clasificador lineal entrenado con el WCB del BCE | 0,3B | MIT | Local | Especialista ligero propio, sin acceso restringido |
| `MoritzLaurer/deberta-v3-large-zeroshot-v2.0` | Clasificador zero-shot (inferencia de lenguaje natural) | 0,4B | MIT | Local | Línea base sin especialización |
| Llama 3.1 8B Instruct | LLM generalista, zero-shot | 8B (4 bits) | Llama 3.1 Community | Local (MLX) | LLM de propósito general |
| Qwen3 8B | LLM generalista, zero-shot | 8B (4 bits) | Apache 2.0 | Local (MLX) | Segundo LLM, de otra familia |

Los tres modelos de `gtfintechlab` tienen acceso restringido con aprobación manual: el benchmark funciona con los candidatos disponibles y los incorpora cuando se concede el acceso. Como referencia externa, el paper del WCB obtiene en frases del BCE un F1 ponderado de 0,755 con RoBERTa-large ajustado, frente a 0,60-0,64 de LLMs grandes en zero-shot (Llama-3-70B, GPT-4o, DeepSeek-V3, Gemini 2.0 Flash), siempre sobre texto escrito. La pregunta es si esa ventaja se mantiene en lenguaje hablado.

Queda fuera FinBERT (`ProsusAI/finbert`): mide sentimiento financiero (positivo/negativo), no postura monetaria, y traducir uno en otro es arbitrario: "inflation rose to 3.0 per cent" es una noticia negativa, pero una señal hawkish.

In [1]:
# autoreload aplica los cambios de src/ sin reiniciar el kernel
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import ROOT, HISTORY_DIR
from src.text.stance_eval import (cargar_wcb, solapamiento, muestra_en_dominio,
                                  crear_excel_etiquetado, leer_etiquetas,
                                  predecir_candidato, cargar_resultados, comprobar_control, evaluar,
                                  comparar_con_mejor, aplicar_regla, metricas_relevancia, matriz_confusion)
from src.text.models.stance_backends import (ClasificadorSecuencia, ClasificadorNLI, ClasificadorEmbeddings,
                                             ClasificadorLLM, hay_acceso, mlx_disponible)

## 3. Datos de referencia

Usamos dos conjuntos complementarios. Ninguno basta por sí solo: el primero es externo y comparable con la literatura, pero de otro registro (texto escrito); el segundo es de nuestro dominio (lenguaje hablado de las ruedas de prensa), pero más pequeño y con un único anotador.

### 3.1 WCB: frases del BCE etiquetadas por expertos

El dataset `gtfintechlab/european_central_bank` (Shah et al., 2025) contiene 1.000 frases de las actas de política monetaria del BCE (2015-2024) etiquetadas como hawkish, dovish, neutral o irrelevant, con tres particiones aleatorias train/val/test (700/150/150), una por semilla. Usamos la de la semilla 5768: el test para evaluar a todos los candidatos, y train y val para entrenar el clasificador lineal sobre embeddings y ajustar su regularización. Las frases del WCB están en minúsculas, así que los modelos entrenados con ellas recibirán también el texto en minúsculas, para no introducir una diferencia entre entrenamiento e inferencia.

Limitaciones: las actas son texto escrito, no lenguaje hablado, y la ficha de los modelos del WCB no documenta con qué semilla se entrenaron, de modo que su resultado en este test puede ser optimista si alguna de sus frases formó parte de su entrenamiento. Por ambas razones, la decisión se basa en la muestra propia (3.2). El paper del WCB reporta F1 ponderado; lo calculamos también para contrastar nuestras cifras con las publicadas.

In [2]:
# Las tres particiones (train, val, test) suman las 1.000 frases del dataset
wcb = cargar_wcb()
print(f"{len(wcb)} frases del WCB | años {wcb['year'].min()}-{wcb['year'].max()}")
pd.crosstab(wcb["split"], wcb["label"], margins=True)

/Users/eduardogonzalezarroyo/Developer/hawk-and-dove_BCE/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1000 frases del WCB | años 2015-2024


label,dovish,hawkish,irrelevant,neutral,All
split,,,,,
test,53,40,5,52,150
train,237,157,17,289,700
val,48,43,2,57,150
All,338,240,24,398,1000


In [3]:
# Control de contaminación: ¿alguna frase del WCB aparece también en nuestras ruedas de prensa?
corpus = pd.read_parquet(HISTORY_DIR / "transcripts.parquet")
frases_wcb_en_corpus = solapamiento(wcb, corpus)
print(f"Frases del WCB presentes en nuestro corpus: {len(frases_wcb_en_corpus)}")

Frases del WCB presentes en nuestro corpus: 6


Si alguna frase coincide, se excluye de la muestra propia, porque los modelos del WCB podrían haberla visto durante su entrenamiento.

### 3.2 Muestra propia de ruedas de prensa

- **Por qué.** Los modelos especializados se entrenaron con texto escrito (actas del BCE o de la Fed) y aquí se aplican a lenguaje hablado: respuestas espontáneas, frases encadenadas, matices. Solo una muestra de nuestro dominio mide lo que de verdad importa para la aplicación.
- **Diseño.** De cada una de las 55 ruedas se eligen al azar una frase de la declaración y tres de las respuestas del panel (presidenta o vicepresidente), con un mínimo de seis palabras: 220 frases. Estratificar por rueda garantiza cubrir todos los regímenes del periodo (relajación en 2019-2021, endurecimiento en 2022-2023, recortes en 2024-2025). Las frases de menos de seis palabras se excluyen porque son fórmulas ("Thank you.") que apenas pesan en la puntuación.
- **Etiquetado a ciegas.** El Excel no muestra la fecha ni el hablante, y las frases van en orden aleatorio, para que el periodo no condicione la etiqueta. La guía adapta al BCE las definiciones del WCB, de modo que el criterio coincide con el que aprendieron los modelos especializados; el prompt de los LLM usa esa misma guía.
- **Limitaciones.** Un único anotador, sin medida de acuerdo entre anotadores. La muestra está estratificada por diseño (una frase de declaración por cada tres de respuestas) y no ponderada por población: sirve para comparar modelos, no para estimar su precisión absoluta en el conjunto del corpus.

#### Guía de etiquetado

¿Qué orientación de política monetaria transmite la frase, leída de forma aislada?

| Etiqueta | Cuándo usarla |
|---|---|
| hawkish | Apunta a una política más restrictiva, o a mantenerla restrictiva, para contener la inflación. Señales: subidas de tipos o sesgo alcista; reducir el balance (fin de compras netas o de reinversiones); inflación alta o al alza; riesgos al alza para la inflación; salarios, empleo o crecimiento fuertes que presionan los precios; subida de los precios de la energía; depreciación del euro. |
| dovish | Apunta a una política más acomodaticia, o a mantenerla acomodaticia. Señales: bajadas de tipos o sesgo bajista; compras de activos, TLTRO u otras medidas de liquidez; inflación baja o a la baja; riesgos a la baja para el crecimiento o la inflación; actividad débil o desempleo al alza; caída de los precios de la energía; apreciación del euro. |
| neutral | Habla de economía o de política monetaria sin una dirección clara: señales en ambos sentidos o equilibradas, compromisos genéricos ("we will follow a data-dependent and meeting-by-meeting approach", "we stand ready to adjust all of our instruments") o descripciones sin implicación para los precios. |
| irrelevant | No aporta información sobre política monetaria ni sobre la economía: saludos, agradecimientos, logística, procedimiento, anécdotas. |

Reglas:

1. Etiqueta solo lo que dice la frase: no uses el contexto ni lo que sabes que pasó después.
2. Los datos económicos cuentan por la presión que ejercen sobre los precios: "inflation rose to 3.0 per cent" → hawkish; "growth slowed" → dovish.
3. Si hay señales en ambos sentidos, elige la dominante; si están equilibradas, neutral.
4. Mantener los tipos, sin más, es neutral; si se añade una orientación ("for as long as necessary", "at their present or lower levels"), etiqueta según esa orientación.
5. Si dudas, elige la etiqueta más probable y escribe "duda" en la columna de comentario.

In [4]:
datos_dir = ROOT / "benchmarks" / "data"
ruta_muestra = datos_dir / "postura_muestra.csv"
ruta_excel = datos_dir / "postura_etiquetado.xlsx"
datos_dir.mkdir(parents=True, exist_ok=True)

# Solo se crean si no existen, para no sobrescribir nunca un etiquetado en curso.
# Para regenerarlos, hay que borrar ambos ficheros.
if ruta_excel.exists():
    print(f"Ya existe {ruta_excel.name}: no se regenera")
    muestra = pd.read_csv(ruta_muestra)
else:
    muestra = muestra_en_dominio(corpus, excluir=frases_wcb_en_corpus)
    muestra.to_csv(ruta_muestra, index=False)
    crear_excel_etiquetado(muestra, ruta_excel)
    print(f"Creados {ruta_muestra.name} y {ruta_excel.name} en {datos_dir}")

print(f"{len(muestra)} frases de {muestra['date'].nunique()} ruedas de prensa")
muestra["role"].value_counts()

Ya existe postura_etiquetado.xlsx: no se regenera
220 frases de 55 ruedas de prensa


role
answer       165
statement     55
Name: count, dtype: int64

#### Comprobación del etiquetado

Lee el Excel y muestra el progreso y la distribución de etiquetas. Puede ejecutarse en cualquier momento.

In [5]:
etiquetas = leer_etiquetas(ruta_excel, ruta_muestra)
n_etiquetadas = etiquetas["label"].notna().sum()
n_dudas = etiquetas["comentario"].astype("string").str.contains("duda", case=False, na=False).sum()
print(f"Etiquetadas: {n_etiquetadas} de {len(etiquetas)} ({n_etiquetadas / len(etiquetas):.0%}) | con duda: {n_dudas}")
pd.crosstab(etiquetas["role"], etiquetas["label"], margins=True)

Etiquetadas: 100 de 220 (45%) | con duda: 0


label,dovish,hawkish,irrelevant,neutral,All
role,,,,,
answer,8,8,28,33,77
statement,3,7,0,13,23
All,11,15,28,46,100


## 4. Métrica

**Tres clases.** La evaluación agrupa "irrelevant" con neutral, igual que la app: esas frases no deben mover la puntuación. Así se penaliza al modelo que atribuye postura a frases ajenas a la política monetaria, un riesgo real en las ruedas de prensa, donde cerca de un tercio de las frases de las respuestas son irrelevantes. Evaluar con cuatro clases sería injusto para FOMC-RoBERTa, que no contempla esa clase, y excluir esas frases ignoraría el riesgo.

**Métrica principal: F1 macro.** Promedia por igual el F1 de hawkish, neutral y dovish, de modo que solo puntúa alto el modelo que detecta las dos clases que transmiten la señal. En la muestra propia, un modelo que respondiera siempre "neutral" obtendría una exactitud de 0,74 y un F1 ponderado de 0,63, pero un F1 macro de 0,28.

**Métricas secundarias**, que describen pero no deciden: F1 por clase; exactitud; correlación de Spearman entre la puntuación (`p_hawkish − p_dovish`) y la etiqueta ordenada (dovish = −1, neutral = 0, hawkish = +1), que es lo que importa para la media de cada rueda; F1 ponderado en cuatro clases, la métrica del paper del WCB; y, en los modelos que contemplan la clase "irrelevant", la detección de frases irrelevantes, de la que sale la columna `relevant`.

**Incertidumbre.** Intervalos al 95 % por bootstrap (2.000 remuestras de frases). Para comparar dos modelos se usa bootstrap pareado: ambos se evalúan sobre las mismas remuestras, de modo que la dificultad de cada frase, común a los dos, no infla la incertidumbre de la diferencia. Con el tamaño de la muestra propia (unas 100 frases, de las que solo una cuarta parte tiene postura), el intervalo del F1 macro de un modelo mide unos ±0,10.

**Regla de decisión.** Decide la muestra propia, que mide el uso real; el WCB es una segunda opinión. Gana el mejor F1 macro, pero entre los modelos estadísticamente empatados con él (el intervalo al 95 % de la diferencia incluye el 0) se elige el más rápido, porque el modelo se aplicará a todo el histórico y a cada rueda nueva.

**Contaminación en el WCB.** El modelo general del WCB se entrenó con el dataset de 25.000 frases anotadas, que incluye las del BCE, y ninguna de las dos fichas documenta qué partición se reservó: en el test del WCB ambos modelos pueden haber visto frases de evaluación, así que su resultado allí es optimista. En la muestra propia no hay contaminación posible para ningún candidato.

## 5. Resultados

### 5.1 Cómo se aplica cada candidato

- **RoBERTa de gtfintechlab.** Reciben el texto en minúsculas, como en su entrenamiento, y su salida se traduce con el orden de etiquetas de su ficha: neutral, hawkish, dovish e irrelevant en los del WCB; dovish, hawkish y neutral en FOMC-RoBERTa.
- **BGE + regresión logística.** Los embeddings no se modifican: solo se entrena una regresión logística multinomial con las 700 frases de train del WCB, con pesos de clase equilibrados para que la clase mayoritaria no domine. La regularización C se elige en val por F1 macro y el modelo final se reentrena con train y val.
- **DeBERTa-v3 zero-shot.** Una hipótesis por etiqueta, redactada con la guía de etiquetado; la puntuación de implicación de las cuatro hipótesis se normaliza con softmax.
- **LLMs (Llama 3.1 8B y Qwen3 8B).** Prompt zero-shot con la guía de etiquetado traducida al inglés y sin ejemplos: unos ejemplos de las actas del WCB introducirían otro registro, y unos de la muestra propia obligarían a retirarlos de la evaluación. No generan texto: se lee la probabilidad que el modelo asigna a cada etiqueta como primer token de su respuesta y se normaliza entre las cuatro, lo que da probabilidades comparables con las del resto de candidatos (pedir al modelo una cifra de confianza produce valores mal calibrados). Las instrucciones, iguales para todas las frases, se procesan una sola vez y se reutilizan desde caché, de modo que cada frase solo cuesta sus propios tokens. Se ejecutan con MLX en versiones de 4 bits (unos 5 GB), que caben con holgura en 24 GB; en 16 bits ocuparían 16 GB solo de pesos.
- **Frases de control.** Antes de evaluar, cada candidato clasifica cuatro frases evidentes (una subida de tipos, una bajada, un compromiso genérico y un saludo) para detectar un mapa de etiquetas mal configurado.

In [6]:
# Conjuntos de evaluación: test del WCB y muestra propia completa
# (se predicen las 220 frases para no repetir nada si se etiquetan más; se evalúan las etiquetadas)
wcb_test = wcb[wcb["split"] == "test"].reset_index(drop=True)
wcb_test.insert(0, "id", range(1, len(wcb_test) + 1))
etiquetadas = etiquetas.dropna(subset=["label"])
conjuntos = {"propia": muestra[["id", "text"]], "wcb": wcb_test[["id", "text"]]}

referencia_propia = etiquetadas[["id", "label"]]
referencia_wcb = wcb_test[["id", "label"]]
print(f"Muestra propia: {len(etiquetadas)} frases etiquetadas de {len(muestra)} | Test del WCB: {len(wcb_test)} frases")

Muestra propia: 100 frases etiquetadas de 220 | Test del WCB: 150 frases


In [7]:
resultados_dir = ROOT / "benchmarks" / "results" / "postura"

# Orden de las salidas de cada RoBERTa (LABEL_0, LABEL_1...), según su ficha en Hugging Face
ETIQUETAS_WCB = ["neutral", "hawkish", "dovish", "irrelevant"]
ETIQUETAS_FOMC = ["dovish", "hawkish", "neutral"]

wcb_train = wcb[wcb["split"] == "train"]
wcb_val = wcb[wcb["split"] == "val"]


def crear_bge(repo):
    """Clasificador lineal sobre embeddings, entrenado con train y val del WCB."""
    clasificador = ClasificadorEmbeddings(repo)
    regularizacion = clasificador.entrenar(wcb_train["text"], wcb_train["label"], wcb_val["text"], wcb_val["label"])
    resultados_dir.mkdir(parents=True, exist_ok=True)
    regularizacion.to_csv(resultados_dir / "bge_regularizacion.csv", index=False)
    return clasificador


# (nombre, repositorio de Hugging Face, cómo se crea el clasificador a partir del repositorio)
CANDIDATOS = [
    ("WCB-BCE (RoBERTa)", "gtfintechlab/model_european_central_bank_stance_label",
     lambda repo: ClasificadorSecuencia(repo, ETIQUETAS_WCB, minusculas=True)),
    ("WCB-general (RoBERTa)", "gtfintechlab/model_WCB_stance_label",
     lambda repo: ClasificadorSecuencia(repo, ETIQUETAS_WCB, minusculas=True)),
    ("FOMC-RoBERTa", "gtfintechlab/FOMC-RoBERTa",
     lambda repo: ClasificadorSecuencia(repo, ETIQUETAS_FOMC, minusculas=True)),
    ("BGE + regresión logística", "BAAI/bge-large-en-v1.5", crear_bge),
    ("DeBERTa-v3 zero-shot", "MoritzLaurer/deberta-v3-large-zeroshot-v2.0", ClasificadorNLI),
    ("Llama 3.1 8B", "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit", ClasificadorLLM),
    ("Qwen3 8B", "mlx-community/Qwen3-8B-4bit", ClasificadorLLM),
]

### 5.2 Disponibilidad

Los modelos con acceso pendiente se omiten. Cuando se conceda el acceso, basta con volver a ejecutar el notebook: los candidatos ya evaluados se reutilizan desde disco y solo se ejecutan los nuevos.

In [8]:
def disponibilidad(repo, crear):
    if crear is ClasificadorLLM and not mlx_disponible():
        return "no (MLX solo funciona en Mac)"
    return "sí" if hay_acceso(repo) else "no (acceso pendiente)"


estado = pd.DataFrame([{"candidato": nombre, "repositorio": repo, "disponible": disponibilidad(repo, crear)}
                       for nombre, repo, crear in CANDIDATOS])
estado

,candidato,repositorio,disponible
0,WCB-BCE (RoBERTa),gtfintechlab/model_european_central_bank_stanc...,no (acceso pendiente)
1,WCB-general (RoBERTa),gtfintechlab/model_WCB_stance_label,no (acceso pendiente)
2,FOMC-RoBERTa,gtfintechlab/FOMC-RoBERTa,no (acceso pendiente)
3,BGE + regresión logística,BAAI/bge-large-en-v1.5,sí
4,DeBERTa-v3 zero-shot,MoritzLaurer/deberta-v3-large-zeroshot-v2.0,sí
5,Llama 3.1 8B,mlx-community/Meta-Llama-3.1-8B-Instruct-4bit,sí
6,Qwen3 8B,mlx-community/Qwen3-8B-4bit,sí


### 5.3 Predicciones

Cada candidato predice las frases de control, la muestra propia y el test del WCB, y guarda probabilidades, tiempo y memoria en `benchmarks/results/postura/`. La primera ejecución tarda del orden de 10-20 minutos, la mayor parte en descargar o cargar modelos y en los dos LLM.

In [9]:
disponibles = set(estado.loc[estado["disponible"] == "sí", "candidato"])
for nombre, repo, crear in CANDIDATOS:
    if nombre in disponibles:
        predecir_candidato(nombre, lambda crear=crear, repo=repo: crear(repo), conjuntos, resultados_dir)

predicciones, tiempos = cargar_resultados(resultados_dir)

BGE + regresión logística: reutilizo pred_bge_regresion_logistica.csv
DeBERTa-v3 zero-shot: reutilizo pred_deberta_v3_zero_shot.csv
Llama 3.1 8B: reutilizo pred_llama_3_1_8b.csv
Qwen3 8B: reutilizo pred_qwen3_8b.csv


### 5.4 Frases de control

Etiqueta que asigna cada modelo a las cuatro frases evidentes. Un error en la subida o la bajada de tipos indicaría un mapa de etiquetas mal configurado; en el compromiso genérico y el saludo caben discrepancias razonables.

In [10]:
comprobar_control(predicciones)

,hawkish (esperada),dovish (esperada),neutral (esperada),irrelevant (esperada)
modelo,,,,
BGE + regresión logística,hawkish,dovish,neutral,irrelevant
DeBERTa-v3 zero-shot,hawkish,dovish,neutral,irrelevant
Llama 3.1 8B,hawkish,dovish,neutral,irrelevant
Qwen3 8B,hawkish,dovish,neutral,irrelevant


### 5.5 Muestra propia (decide)

In [11]:
tabla_propia = evaluar(predicciones, referencia_propia, "propia")
tabla_propia.round(3)

,modelo,n,f1_macro,ic95_inf,ic95_sup,f1_hawkish,f1_neutral,f1_dovish,exactitud,spearman,f1_ponderado_4
0,Qwen3 8B,100,0.671,0.522,0.795,0.629,0.886,0.500,0.81,0.567,0.576
1,DeBERTa-v3 zero-shot,100,0.645,0.508,0.761,0.625,0.867,0.444,0.79,0.534,0.567
2,Llama 3.1 8B,100,0.573,0.454,0.689,0.667,0.886,0.167,0.81,0.484,0.556
3,BGE + regresión logística,100,0.531,0.403,0.644,0.467,0.819,0.308,0.70,0.202,0.508


### 5.6 Test del WCB (segunda opinión)

El F1 ponderado en cuatro clases es comparable con el del paper (0,755 con RoBERTa-large ajustado en frases del BCE).

In [12]:
tabla_wcb = evaluar(predicciones, referencia_wcb, "wcb")
tabla_wcb.round(3)

,modelo,n,f1_macro,ic95_inf,ic95_sup,f1_hawkish,f1_neutral,f1_dovish,exactitud,spearman,f1_ponderado_4
0,BGE + regresión logística,150,0.619,0.532,0.694,0.563,0.649,0.643,0.627,0.501,0.586
1,Qwen3 8B,150,0.595,0.505,0.674,0.581,0.679,0.526,0.620,0.500,0.564
2,Llama 3.1 8B,150,0.493,0.406,0.571,0.542,0.609,0.328,0.533,0.394,0.466
3,DeBERTa-v3 zero-shot,150,0.338,0.260,0.419,0.157,0.560,0.297,0.427,0.073,0.321


### 5.7 Comparación con el mejor modelo y regla de decisión

Diferencia de F1 macro entre el mejor modelo de la muestra propia y cada uno de los demás, con su intervalo al 95 % por bootstrap pareado. Hay empate cuando el intervalo incluye el 0.

In [13]:
mejor = tabla_propia.loc[0, "modelo"]
comparacion = comparar_con_mejor(predicciones, referencia_propia, "propia", mejor)
recomendado = aplicar_regla(tabla_propia, comparacion, tiempos)

print(f"Mejor F1 macro en la muestra propia: {mejor}")
print(f"Candidato según la regla de decisión: {recomendado}")
comparacion.round(3)

Mejor F1 macro en la muestra propia: Qwen3 8B
Candidato según la regla de decisión: BGE + regresión logística


,modelo,diferencia,ic95_inf,ic95_sup,empate
0,BGE + regresión logística,0.140,-0.060,0.325,True
1,DeBERTa-v3 zero-shot,0.026,-0.136,0.185,True
2,Llama 3.1 8B,0.098,-0.097,0.265,True


In [14]:
# Errores del candidato recomendado (filas: etiqueta de referencia; columnas: predicha)
matriz_confusion(predicciones, referencia_propia, "propia", recomendado)

predicha,dovish,hawkish,neutral
referencia,,,
dovish,4,1,6
hawkish,3,7,5
neutral,8,7,59


### 5.8 Detección de frases irrelevantes

Solo en los modelos que contemplan la clase "irrelevant". Es la base de la columna `relevant` de `stance.csv`: las frases irrelevantes no cuentan en la puntuación de la rueda.

In [15]:
metricas_relevancia(predicciones, referencia_propia, "propia").round(3)

,modelo,precision,recall,f1
0,Qwen3 8B,0.536,0.536,0.536
1,DeBERTa-v3 zero-shot,0.619,0.464,0.531
2,BGE + regresión logística,0.786,0.393,0.524
3,Llama 3.1 8B,0.647,0.393,0.489


### 5.9 Regularización del clasificador sobre embeddings

F1 macro en val del WCB para cada valor de C. El elegido se reentrena con train y val.

In [16]:
ruta_regularizacion = resultados_dir / "bge_regularizacion.csv"
pd.read_csv(ruta_regularizacion).round(3) if ruta_regularizacion.exists() else "BGE no se ha ejecutado"

,C,f1_macro_val,elegido
0,0.01,0.579,False
1,0.10,0.577,False
2,1.00,0.657,False
3,10.00,0.700,True
4,100.00,0.684,False


## 6. Latencia y memoria

Medidas en el Mac de desarrollo (M4, 24 GB) sin contar la carga del modelo. La memoria es la de los pesos en los modelos de PyTorch y el pico de memoria en los de MLX. El tiempo estimado del histórico es el de clasificar todas las frases del panel de las 55 ruedas. En los LLM, `masa_etiquetas` es la probabilidad media que el modelo deja en las cuatro etiquetas: si fuera baja, el modelo no estaría siguiendo el formato de respuesta y sus probabilidades serían poco fiables.

In [17]:
n_historico = len(corpus[corpus["speaker"].isin(["presidenta", "vicepresidente"])
                         & corpus["role"].isin(["statement", "answer"])])
latencia = tiempos.assign(minutos_historico=(n_historico / tiempos["frases_por_segundo"] / 60).round(1))
print(f"Frases del panel en el histórico: {n_historico}")
latencia

Frases del panel en el histórico: 12276


,modelo,frases,segundos,frases_por_segundo,memoria_gb,masa_etiquetas,minutos_historico
0,BGE + regresión logística,370,4.8,77.41,1.34,NaN,2.6
1,DeBERTa-v3 zero-shot,370,41.0,9.02,0.87,NaN,22.7
2,Llama 3.1 8B,370,132.0,2.80,4.92,0.997,73.1
3,Qwen3 8B,370,154.1,2.40,5.07,1.000,85.2


In [18]:
# Tablas para el README
tabla_propia.to_csv(resultados_dir / "metricas_propia.csv", index=False)
tabla_wcb.to_csv(resultados_dir / "metricas_wcb.csv", index=False)
comparacion.to_csv(resultados_dir / "comparacion_con_mejor.csv", index=False)
print(f"Tablas guardadas en {resultados_dir}")

Tablas guardadas en /Users/eduardogonzalezarroyo/Developer/hawk-and-dove_BCE/benchmarks/results/postura


## 7. Decisión

**Modelo elegido: Qwen3 8B** (zero-shot, MLX en 4 bits). La etiqueta en tres clases, la puntuación y la relevancia se derivan de sus cuatro probabilidades.

**Evidencia.**
- **Muestra propia (decide).** Qwen3 8B obtiene el mejor F1 macro (0,671; IC 95 % 0,52-0,80) y la mejor correlación entre puntuación y etiqueta (Spearman 0,57). Le siguen DeBERTa-v3 zero-shot (0,645), Llama 3.1 8B (0,573) y el clasificador sobre embeddings (0,531; Spearman 0,20).
- **Test del WCB (segunda opinión).** Qwen3 8B es segundo (0,595), cerca del clasificador sobre embeddings (0,619), que juega en casa porque se entrenó con frases del mismo dataset. Qwen3 8B es el único candidato que queda entre los dos mejores en ambos conjuntos.
- **Contraste con la literatura.** En el WCB, Qwen3 8B alcanza un F1 ponderado de 0,564, en línea con los 0,60-0,64 que el paper obtiene en zero-shot con LLMs mucho mayores. El clasificador sobre embeddings (0,586) queda por debajo de los RoBERTa ajustados del paper (0,72-0,76), como cabe esperar de un modelo que no ajusta sus embeddings.
- **Controles.** Todos los candidatos aciertan las frases de control. Los LLM dejan casi toda la probabilidad en las cuatro etiquetas (masa ≥ 0,997), es decir, siguen el formato de respuesta y sus probabilidades son fiables.

**Corrección de la regla de decisión.** Aplicada al pie de la letra, la regla de la sección 4 elige el clasificador sobre embeddings. Con 100 frases etiquetadas, de las que solo 26 tienen postura, el intervalo de la diferencia con el mejor incluye el 0 para todos los candidatos, y el desempate por velocidad favorece al más rápido. Pero una diferencia no significativa con tan poca potencia no demuestra que los modelos sean equivalentes: el clasificador sobre embeddings es el peor en la muestra que decide (0,14 menos de F1 macro que el mejor) y su puntuación apenas sigue a la etiqueta en lenguaje hablado.

Para preferir un modelo más rápido habría que demostrar que no es peor que el mejor más allá de un margen tolerable (no inferioridad), es decir, que el límite superior del intervalo de la diferencia quede por debajo de ese margen. Ningún candidato lo cumple: sus límites superiores van de 0,19 a 0,33, así que la conclusión no depende del margen elegido. Por tanto, se elige la mejor estimación puntual. Además, el argumento de la velocidad pierde fuerza a la vista de las latencias medidas: el candidato más lento clasifica las 12.276 frases del histórico en unos 85 minutos y una rueda nueva en un par de minutos.

**Por qué no los demás.**
- **DeBERTa-v3 zero-shot.** Empata con Qwen3 en lenguaje hablado, pero se desploma en el texto escrito del WCB (F1 macro 0,34; Spearman 0,07). La declaración inicial, un tercio de las frases de cada rueda, se lee en registro escrito, así que esa inestabilidad es un riesgo real. Una explicación plausible es que la inferencia de lenguaje natural es literal: una frase en estilo indirecto de las actas ("members noted that inflation had risen") no implica la hipótesis de que la política se endurecerá.
- **Clasificador sobre embeddings.** Es el mejor en el WCB, de donde salen sus datos de entrenamiento, y el peor en las ruedas de prensa: aprender de las actas no se transfiere bien al lenguaje hablado.
- **Llama 3.1 8B.** Queda por debajo de Qwen3 en ambos conjuntos y apenas detecta la clase dovish en la muestra propia (F1 0,17).

**Limitaciones.**
- En la muestra propia, la ventaja de Qwen3 sobre DeBERTa y Llama no es estadísticamente concluyente. La decisión se apoya en la mejor estimación, en la consistencia entre conjuntos y en la calidad de la puntuación.
- Hay un único anotador y 100 frases etiquetadas. Etiquetar las 120 restantes (las predicciones ya están calculadas) estrecharía los intervalos a unos ±0,07.
- La detección de frases irrelevantes es la parte más débil de todos los candidatos (F1 ≈ 0,5). Cómo se usa en la puntuación de cada rueda se decide en la fase siguiente.
- Los modelos de gtfintechlab no se han podido evaluar por falta de acceso. Si llega, se incorporan volviendo a ejecutar el notebook y se aplican los mismos criterios.
- MLX solo funciona en Mac. No afecta a la app, porque la postura se precalcula; en otros sistemas, el mismo modelo (`Qwen/Qwen3-8B`) puede ejecutarse con transformers en una GPU.

Modelo registrado en `config.yaml`: `stance: mlx-community/Qwen3-8B-4bit`.